In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms

In [ ]:
batch_size = 256

trainset = datasets.MNIST("./dataset/", train=True, transform = transforms.ToTensor(), download = True)
trainloader = torch.utils.data.DataLoader(trainset, batch_size = batch_size, shuffle = True, num_workers = 2)

testset = datasets.MNIST("../dataset/", train=False, transform = transforms.ToTensor(), download =   True)
testloader = torch.utils.data.DataLoader(testset, batch_size = batch_size, shuffle = False, num_workers = 2)

100%|██████████| 9.91M/9.91M [00:00<00:00, 11.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 342kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.97MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.24MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 9.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 337kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.94MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 946kB/s]


In [ ]:
class SAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder1 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(28 * 28, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, 2000),
            nn.ReLU(),
            nn.Linear(2000, 10),
            nn.ReLU()
        )

        self.decoder1 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(10, 2000),
            nn.Tanh(),
            nn.Linear(2000, 500),
            nn.Tanh(),
            nn.Linear(500, 500),
            nn.Tanh(),
            nn.Linear(500, 28 * 28),
            nn.Tanh()
        )

        self.encoder2 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(28 * 28, 500),
            nn.Tanh(),
            nn.Linear(500, 500),
            nn.Tanh(),
            nn.Linear(500, 2000),
            nn.Tanh(),
            nn.Linear(2000, 10),
            nn.Tanh()
        )

        self.decoder2 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(10, 2000),
            nn.ReLU(),
            nn.Linear(2000, 500),
            nn.ReLU(),
            nn.Linear(500, 500),
            nn.ReLU(),
            nn.Linear(500, 28 * 28),
            nn.ReLU()
        )

    def forward(self, x):
        encode1 = self.encoder1(x)
        decode1 = self.decoder1(encode1)
        encode2 = self.encoder2(decode1)
        decode2 = self.decoder2(encode2)

        return decode2
    
    def forward_layer1(self, x):
        encode1 = self.encoder1(x)
        decode1 = self.decoder1(encode1)

        return decode1
    
    def forward_layer2(self, x):
        encode1 = self.encoder1(x)
        decode1 = self.decoder1(encode1)
        encode2 = self.encoder2(decode1)
        decode2 = self.decoder2(encode2)

        return decode2

In [ ]:
class SAE(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder_layer_1 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(28 * 28, 500),
            nn.ReLU(),
        )

        self.decoder_layer_1 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(500, 28 * 28),
            nn.Tanh(),
        )

        self.encoder_layer_2 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(500, 500),
            nn.ReLU(),
        )

        self.decoder_layer_2 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(500, 500),
            nn.ReLU(),
        )

        self.encoder_layer_3 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(500, 2000),
            nn.ReLU(),
        )

        self.decoder_layer_3 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(2000, 500),
            nn.ReLU(),
        )

        self.encoder_layer_4 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(2000, 10),
            nn.Tanh(),
        )

        self.decoder_layer_4 =  nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(10, 2000),
            nn.ReLU(),
        )

    def forward(self, x):
        encoded = self.encoder_layer_4(
                    self.encoder_layer_3(
                        self.encoder_layer_2(
                            self.encoder_layer_1(x)
                        )
                    )
                )

        decoded = self.decoder_layer_1(
                    self.decoder_layer_2(
                        self.decoder_layer_3(
                            self.decoder_layer_4(encoded)
                        )
                    )
                )
        
        return decoded
    
    def forward_layer_1(self, x):
        encode1 = self.encoder_layer_1(x)
        decode1 = self.decoder_layer_1(encode1)

        return decode1
    
    def forward_layer_2(self, x):
        encode1 = self.encoder_layer_1(x)
        encode2 = self.encoder_layer_2(encode1)

        decode2 = self.decoder_layer_2(encode2)
        decode1 = self.decoder_layer_1(decode2)

        return decode1
    
    def forward_layer_3(self, x):
        encode1 = self.encoder_layer_1(x)
        encode2 = self.encoder_layer_2(encode1)
        encode3 = self.encoder_layer_2(encode2)

        decode3 = self.decoder_layer_2(encode3)
        decode2 = self.decoder_layer_2(decode3)
        decode1 = self.decoder_layer_1(decode2)

        return decode1
    
    def forward_layer_4(self, x):
        encode1 = self.encoder_layer_1(x)
        encode2 = self.encoder_layer_2(encode1)
        encode3 = self.encoder_layer_2(encode2)
        encode4 = self.encoder_layer_2(encode3)

        decode4 = self.decoder_layer_2(encode4)
        decode3 = self.decoder_layer_2(decode4)
        decode2 = self.decoder_layer_2(decode3)
        decode1 = self.decoder_layer_1(decode2)

        return decode1

# SAE Training

In [ ]:
model = SAE()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pretrain_epochs = 20
train_epochs = 50


# Pretraining

# Pretrain layer 1
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.train()
for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer1(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

# Pretrain layer 2
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.encoder1.requires_grad = False
model.decoder1.requires_grad = False

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer_2(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")


# Pretrain layer 3
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.encoder1.requires_grad = False
model.decoder1.requires_grad = False
model.encoder2.requires_grad = False
model.decoder2.requires_grad = False

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer_3(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")


# Pretrain layer 4
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.encoder1.requires_grad = False
model.decoder1.requires_grad = False
model.encoder2.requires_grad = False
model.decoder2.requires_grad = False
model.encoder3.requires_grad = False
model.decoder3.requires_grad = False

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer_4(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")


# Training
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

train_losses = []

model.encoder1.requires_grad = True
model.decoder1.requires_grad = True
model.encoder2.requires_grad = True
model.decoder2.requires_grad = True
model.encoder3.requires_grad = True
model.decoder3.requires_grad = True

model.eval() # dropout only for pretraining
for epoch in range(train_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

In [ ]:
model = SAE()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

pretrain_epochs = 20
train_epochs = 50


# Pretraining

# Pretrain layer 1
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.train()
for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer_1(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

# Pretrain layer 2
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

pretrain_losses = []

model.encoder1.requires_grad = False
model.decoder1.requires_grad = False

for epoch in range(pretrain_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model.forward_layer2(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        pretrain_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")
      
# Training
loss_function = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.1, momentum = 0.9)
scheduler = optim.StepLR(
    optimizer, 
    step_size = 20000, #Amount of epochs the scheduler will update the lr
    gamma = 0.1 #by how much the scheduler will divide the current lr
    )

train_losses = []

model.encoder1.requires_grad = True
model.decoder1.requires_grad = True
model.eval() # dropout only for pretraining
for epoch in range(train_epochs):
    for images, _ in trainloader:

        images = torch.flatten(images)
        reconstructed = model(images)
        loss = loss_function(reconstructed, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    print(f"Pretrain: Epoch {epoch+1}/{pretrain_epochs}, Loss: {loss.item():.6f}")

# Clustering